In [ ]:
CHUNK_SIZE=500
CHUNK_OVERLAP=100

In [2]:
query_list = [
    "연차를 사용하려면 최소 며칠 전에 신청해야 하나요?",
    "서울 출장 시 숙박비는 최대 얼마까지 지원되나요?",
    "출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?",
    "회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?",
    "재택근무를 신청하려면 어떤 조건을 충족해야 하나요?",
    "직원이 업무 관련 교육을 받을 경우 회사에서 지원하는 비용 기준은 어떻게 되나요?",
    "출장 중 택시비를 회사 경비로 처리할 수 있는 경우는 언제인가요?",
    "개인 사정으로 출근 시간이 늦어질 경우 어떤 방식으로 보고해야 하나요?",
    "업무 중 시스템 장애가 발생했을 때 가장 먼저 해야 하는 조치는 무엇인가요?",
    "제주도로 2박 3일 출장을 가는 경우 숙박비 지원 한도와 출장비 정산 기한을 각각 알려주세요.",
]

In [3]:
from dotenv import load_dotenv 
import os

load_dotenv()
os.environ["LANGSMITH_PROJECT"] = "RAG_02"

In [15]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1} (p.{d.metadata.get("page_label")}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )

In [4]:
from langchain_community.document_loaders import PyPDFLoader 
from langchain_community.vectorstores import FAISS 
from langchain_openai import OpenAIEmbeddings 
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF 로드 
loader = PyPDFLoader("data/가상Tech_업무가이드.pdf")
documents = loader.load() 

# 문서 분할 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

split_docs = text_splitter.split_documents(documents) 

# 임베딩 
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# FAISS 벡터스토어 생성 
db = FAISS.from_documents(split_docs, embeddings)

C:\Users\user\AppData\Local\Temp\ipykernel_11780\2884977073.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
# 리트리버 생성
retriever = db.as_retriever(search_kwargs={"k":7})

In [ ]:
# 허깅페이스 모델 한 번만 불러오기
from langchain_community.cross_encoders import HuggingFaceCrossEncoder 
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

In [16]:
from langchain_classic.retrievers import ContextualCompressionRetriever 
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker 

compressor = CrossEncoderReranker(model=model, top_n=3)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever,
)

for i, query in enumerate(query_list):
    compressed_docs = compression_retriever.invoke(query)
    print(f"\n질문{i+1}. {query}\n")
    pretty_print_docs(compressed_docs)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


질문1. 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?

Document 1 (p.8):

가상Tech 업무 가이드
사내용 · v1.0
문서 책임: People Operations
08
CHAPTER 07
휴가·비용·복지
예측 가능한 신청 절차로 개인의 휴식과 팀 운영을 함께 지킵니다.
 
7.1 휴가
 구분
 신청 시점
 승인·증빙
연차
1일 전
팀장 승인, 3일 이상은 2주 전 협의
반차
당일 10:00 전
팀장 승인
병가
가능한 즉시
3일 초과 시 진료 확인서 제출
경조휴가
사유 확인 즉시
People Ops 확인, 사내 규정 일수 적용
7.2 비용 처리
 • 법인카드 사용 내역은 영수증과 목적을 지출일로부터 5영업일 안에 ExpenseHub에 등록합니다.
 • 건당 10만원 이상 소프트웨어·비품은 구매 전에 팀장 승인을 받습니다. 연간 구독은 IT 자산 검토도 필요합니다.
 • 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
----------------------------------------------------------------------------------------------------
Document 2 (p.8):

• 국내 출장 교통·숙박은 출발 5영업일 전 신청합니다. 숙박 한도는 1박 15만원(세금 포함)입니다.
 • 택시는 23:00 이후 퇴근, 대중교통 미운행, 고객 일정 등 업무상 필요가 있을 때 실비 지원합니다.
 
7.3 주요 복지
 제도
 지원 내용
 신청
성장 지원
연 120만원 교육·도서·컨퍼런스
LearnHub에서 사전 승인
건강 지원
연 60만원 운동·상담
월별 영수증 제출
장비 지원
3년 주기 노트북 교체, 모니터 1대
#help-it 티켓
리프레시
근속 3년마다 유급 10영업일
3개월 전 팀과 일정 협의
휴가 중 연락
휴가자는 업무 메시지에 응답할 의무가 없습니다. 인수인계 문서에 대체 담당자,
진행 상태, 긴급 연락 기준을 남기고 Slack 상태와

답변 생성

In [24]:
from langchain_core.prompts import ChatPromptTemplate 
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    context = f"\n{'-' * 100}\n".join(
        [f"Document {i+1} (p.{d.metadata.get("page_label")}):\n\n" + d.page_content for i, d in enumerate(docs)]
        )
    return context
    
prompt = ChatPromptTemplate.from_messages([
    ("system", """
    1. 당신은 사용자의 질문에 대답을 하는 친절한 AI agent이다.
    2. 반드시 문서에 있는 내용만으로 답한다.
    3. 사용자의 질문을 문서에서 찾을 수 없으면 문서에서 찾지 못하여 대답을 할 수 없다고 답변한다.
    4. 질문이 여러 항목이면 항목마다 문서에 있는지 판단하고, 없는 항목은 '문서에 없음'이라고 밝힌다.
    5. 답변에 근거 페이지를 밝힌다.

    문서:
    {context}"""),
    ("human", "{question}"),
])

llm = ChatOpenAI(model="gpt-5.6-luna", temperature=0)

rag_chain = prompt | llm | StrOutputParser()

for i, query in enumerate(query_list):
    compressed_docs = compression_retriever.invoke(query)
    context = format_docs(compressed_docs)
    answer = rag_chain.invoke({"context": context, "question": query})
    print(f"질문 {i+1}: {query}\n\n답변: {answer}\n\n{'-'*80}\n\n")

질문 1: 연차를 사용하려면 최소 며칠 전에 신청해야 하나요?

답변: 연차는 **최소 1일 전**에 신청해야 합니다. 단, **3일 이상 사용 시에는 2주 전 협의**가 필요합니다.  
근거: **문서 1, p.8**

--------------------------------------------------------------------------------


질문 2: 서울 출장 시 숙박비는 최대 얼마까지 지원되나요?

답변: 서울 출장 숙박비는 1박 최대 15만 원(세금 포함)까지 지원됩니다. 국내 출장·숙박은 출발 5영업일 전에 신청해야 합니다.  
(근거: Document 1·2, p.8)

--------------------------------------------------------------------------------


질문 3: 출장을 다녀온 후 비용 정산은 언제까지 완료해야 하나요?

답변: 문서에는 **출장 후 비용 정산을 완료해야 하는 별도 기한은 명시되어 있지 않습니다.**

다만 법인카드 사용 내역은 **지출일로부터 5영업일 이내에 영수증과 사용 목적을 ExpenseHub에 등록**해야 합니다.  
**근거: p.8**

--------------------------------------------------------------------------------


질문 4: 회사 노트북을 분실했을 경우 어떤 절차로 신고해야 하나요?

답변: 문서에는 **회사 노트북 분실 시 신고 절차가 명시되어 있지 않아** 정확한 절차를 답변할 수 없습니다.

다만 문서에는 다음 내용이 있습니다.

- 업무는 회사 관리 기기에서만 수행합니다.
- 계정 탈취가 의심되면 즉시 네트워크를 끊고 **02-555-0142** 또는 **#security-incident**로 신고합니다. *(Document 2, p.3)*
- 기기는 마지막 근무일 17:00까지 IT 자산 담당자에게 반납해야 합니다. *(Document 1, p.3)*

-